In [4]:
import os

os.chdir(r"C:\Users\BANAPURAM VINAY TEJA\OneDrive\Desktop\ROAD-PROJ")

print("Current directory:", os.getcwd())
print("Project files:", os.listdir())

Current directory: C:\Users\BANAPURAM VINAY TEJA\OneDrive\Desktop\ROAD-PROJ
Project files: ['.git', 'dft-road-casualty-statistics-casualty-last-5-years.csv', 'dft-road-casualty-statistics-collision-last-5-years.csv', 'dft-road-casualty-statistics-vehicle-last-5-years(1).csv', 'hospital-finder.html', 'ROAD.ipynb', 'roadguard_bundle.pkl', 'roadguard_preprocessor.pkl', 'roadguard_rf_model.pkl', 'STATS20_2024_specification.pdf']


In [5]:
import joblib

model = joblib.load("roadguard_rf_model.pkl")
preprocessor = joblib.load("roadguard_preprocessor.pkl")

print("Model loaded:", type(model))
print("Preprocessor loaded:", type(preprocessor))

Model loaded: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Preprocessor loaded: <class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [6]:
print("Expected input columns:")
print(preprocessor.feature_names_in_.tolist())

print("\nNumber of input features:",
      len(preprocessor.feature_names_in_))

Expected input columns:
['longitude', 'latitude', 'day_of_week', 'first_road_class', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'second_road_class', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area', 'trunk_road_flag', 'year', 'month', 'hour']

Number of input features: 18


In [7]:
def predict_severity(input_data):

    # Convert input dictionary into DataFrame
    input_df = pd.DataFrame([input_data])

    # Ensure correct column order
    input_df = input_df[preprocessor.feature_names_in_]

    # Apply saved preprocessing
    processed_data = preprocessor.transform(input_df)

    # Predict using Random Forest
    prediction = model.predict(processed_data)

    # Convert numerical prediction into label
    severity_labels = {
        1: "Fatal",
        2: "Serious",
        3: "Slight"
    }

    return severity_labels[prediction[0]]

In [9]:
accident_data = {
    "longitude": -0.135,
    "latitude": 51.507,
    "day_of_week": 2,
    "first_road_class": 3,
    "road_type": 6,
    "speed_limit": 30,
    "junction_detail": 3,
    "junction_control": 2,
    "second_road_class": 6,
    "pedestrian_crossing": 0,
    "light_conditions": 1,
    "weather_conditions": 1,
    "road_surface_conditions": 1,
    "urban_or_rural_area": 1,
    "trunk_road_flag": 0,
    "year": 2024,
    "month": 6,
    "hour": 14
}

In [10]:
predict_severity(accident_data)

'Slight'

In [11]:
feature_names = preprocessor.feature_names_in_.tolist()

print("Total input features:", len(feature_names))

for index, feature in enumerate(feature_names, start=1):
    print(f"{index}. {feature}")

Total input features: 18
1. longitude
2. latitude
3. day_of_week
4. first_road_class
5. road_type
6. speed_limit
7. junction_detail
8. junction_control
9. second_road_class
10. pedestrian_crossing
11. light_conditions
12. weather_conditions
13. road_surface_conditions
14. urban_or_rural_area
15. trunk_road_flag
16. year
17. month
18. hour


In [12]:
import json

with open("roadguard_features.json", "w") as file:
    json.dump(feature_names, file, indent=4)

print("Feature documentation saved!")

Feature documentation saved!


In [13]:
with open("roadguard_features.json", "r") as file:
    print(file.read())

[
    "longitude",
    "latitude",
    "day_of_week",
    "first_road_class",
    "road_type",
    "speed_limit",
    "junction_detail",
    "junction_control",
    "second_road_class",
    "pedestrian_crossing",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "urban_or_rural_area",
    "trunk_road_flag",
    "year",
    "month",
    "hour"
]


In [14]:
print("Preprocessor input columns:")
print(preprocessor.feature_names_in_.tolist())

print("\nCategorical columns:")
print(preprocessor.transformers_)

Preprocessor input columns:
['longitude', 'latitude', 'day_of_week', 'first_road_class', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'second_road_class', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area', 'trunk_road_flag', 'year', 'month', 'hour']

Categorical columns:
[('categorical', OneHotEncoder(handle_unknown='ignore'), ['first_road_class', 'road_type', 'junction_detail', 'junction_control', 'second_road_class', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area', 'trunk_road_flag']), ('numerical', FunctionTransformer(accept_sparse=True, check_inverse=False,
                    feature_names_out='one-to-one'), ['longitude', 'latitude', 'speed_limit', 'year', 'month', 'hour', 'day_of_week'])]


In [15]:
categorical_encoder = preprocessor.named_transformers_["categorical"]

for column, categories in zip(
    preprocessor.transformers_[0][2],
    categorical_encoder.categories_
):
    print(f"\n{column}:")
    print(categories)


first_road_class:
[-1  1  2  3  4  5  6]

road_type:
[1 2 3 6 7 9]

junction_detail:
[-1  0 13 16 17 18 19 99]

junction_control:
[-1  1  2  3  4  9]

second_road_class:
[-1  0  1  2  3  4  5  6  9]

pedestrian_crossing:
[-1  0 11 12 13 14 15 16 17 99]

light_conditions:
[1. 4. 5. 6. 7.]

weather_conditions:
[1. 2. 3. 4. 5. 6. 7. 8. 9.]

road_surface_conditions:
[1. 2. 3. 4. 5. 9.]

urban_or_rural_area:
[1 2 3]

trunk_road_flag:
[-1  1  2]


In [16]:
import json

categorical_encoder = preprocessor.named_transformers_["categorical"]

category_mapping = {}

categorical_columns = preprocessor.transformers_[0][2]

for column, categories in zip(
    categorical_columns,
    categorical_encoder.categories_
):
    category_mapping[column] = categories.tolist()

with open("roadguard_category_mappings.json", "w") as file:
    json.dump(category_mapping, file, indent=4)

print("Category mappings saved successfully!")

Category mappings saved successfully!


In [17]:
import os

files = [
    "roadguard_rf_model.pkl",
    "roadguard_preprocessor.pkl",
    "roadguard_bundle.pkl",
    "roadguard_features.json",
    "roadguard_category_mappings.json",
    "README.md"
]

for file in files:
    print(file, "->", os.path.exists(file))

roadguard_rf_model.pkl -> True
roadguard_preprocessor.pkl -> True
roadguard_bundle.pkl -> True
roadguard_features.json -> True
roadguard_category_mappings.json -> True
README.md -> False


In [18]:
readme_content = """# RoadGuard AI - Machine Learning Module

## Overview
Random Forest model for predicting road collision severity.

## Predictions
1 - Fatal
2 - Serious
3 - Slight

## Model Files
- roadguard_rf_model.pkl
- roadguard_preprocessor.pkl
- roadguard_bundle.pkl
- roadguard_features.json
- roadguard_category_mappings.json

## Input
The model expects 18 input features.
The preprocessor transforms them into 79 processed features.

## Important
This is an experimental collision severity model, not a safety-critical system.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content)

print("README created:", os.path.exists("README.md"))

README created: True


In [19]:
import os

for file in os.listdir():
    if file.startswith("roadguard") or file == "README.md":
        print(file, "->", os.path.getsize(file) / (1024 * 1024), "MB")

README.md -> 0.0005168914794921875 MB
roadguard_bundle.pkl -> 289.3909912109375 MB
roadguard_category_mappings.json -> 0.001247406005859375 MB
roadguard_features.json -> 0.0003814697265625 MB
roadguard_preprocessor.pkl -> 0.0038957595825195312 MB
roadguard_rf_model.pkl -> 620.4958505630493 MB


In [21]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.feature_names_in_ = X.columns
        self.frequency_maps_ = {}

        for col in X.columns:
            values = X[col].astype("string").fillna("__MISSING__")
            self.frequency_maps_[col] = (
                values.value_counts(normalize=True).to_dict()
            )

        return self

    def transform(self, X):
        X = pd.DataFrame(
            X,
            columns=self.feature_names_in_
        ).copy()

        encoded = pd.DataFrame(index=X.index)

        for col in X.columns:
            values = X[col].astype("string").fillna("__MISSING__")
            encoded[col] = values.map(
                self.frequency_maps_[col]
            ).fillna(0)

        return encoded.to_numpy()

In [22]:
import joblib

bundle = joblib.load("roadguard_bundle.pkl")

print("Bundle type:", type(bundle))

if isinstance(bundle, dict):
    print("Bundle contents:", bundle.keys())

Bundle type: <class 'dict'>
Bundle contents: dict_keys(['model', 'encoder', 'string_columns', 'feature_columns', 'target_labels'])


In [23]:
model = bundle["model"]
encoder = bundle["encoder"]

string_columns = bundle["string_columns"]
feature_columns = bundle["feature_columns"]
target_labels = bundle["target_labels"]

print("Model:", type(model))
print("Encoder:", type(encoder))
print("String columns:", string_columns)
print("Number of expected features:", len(feature_columns))
print("Target labels:", target_labels)

Model: <class 'sklearn.pipeline.Pipeline'>
Encoder: <class 'sklearn.preprocessing._encoders.OrdinalEncoder'>
String columns: ['local_authority_ons_district', 'local_authority_highway_current', 'special_conditions_at_site', 'carriageway_hazards_historic', 'carriageway_hazards', 'lsoa_of_accident_location']
Number of expected features: 32
Target labels: {1: 'Fatal', 2: 'Serious', 3: 'Slight'}


In [24]:
print("Model pipeline:")
print(model)

print("\nModel expected input columns:")
print(model.feature_names_in_)

print("\nNumber of input columns:", len(model.feature_names_in_))

Model pipeline:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['longitude', 'latitude',
                                                   'number_of_vehicles',
                                                   'number_of_casualties',
                                                   'speed_limit']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle

In [26]:
sample_input = pd.DataFrame(
    [{col: 0 for col in feature_columns}]
)

print(sample_input)

   longitude  latitude  police_force  number_of_vehicles  \
0          0         0             0                   0   

   number_of_casualties  day_of_week  local_authority_district  \
0                     0            0                         0   

   local_authority_ons_district  local_authority_highway_current  \
0                             0                                0   

   first_road_class  ...  special_conditions_at_site  \
0                 0  ...                           0   

   carriageway_hazards_historic  carriageway_hazards  urban_or_rural_area  \
0                             0                    0                    0   

   did_police_officer_attend_scene_of_accident  trunk_road_flag  \
0                                            0                0   

   lsoa_of_accident_location  year  month  hour  
0                          0     0      0     0  

[1 rows x 32 columns]


In [27]:
# Apply the saved encoder to the six string columns
sample_processed = sample_input.copy()

sample_processed[string_columns] = encoder.transform(
    sample_processed[string_columns]
)

# Ensure the expected column order
sample_processed = sample_processed[feature_columns]

# Make prediction
prediction = model.predict(sample_processed)

print("Predicted severity:", target_labels[prediction[0]])

TypeError: '<' not supported between instances of 'int' and 'str'

In [28]:
preprocessor = model.named_steps["preprocessor"]

categorical_pipeline = preprocessor.named_transformers_["categorical"]
onehot = categorical_pipeline.named_steps["onehot"]

categorical_columns = preprocessor.transformers_[1][2]

for col, categories in zip(categorical_columns, onehot.categories_):
    types = set(type(value).__name__ for value in categories)
    print(col, "->", types)

first_road_class -> {'float64'}
road_type -> {'float64'}
junction_detail_historic -> {'float64'}
junction_detail -> {'float64'}
junction_control -> {'float64'}
second_road_class -> {'float64'}
pedestrian_crossing_human_control_historic -> {'float64'}
pedestrian_crossing_physical_facilities_historic -> {'float64'}
pedestrian_crossing -> {'float64'}
light_conditions -> {'float64'}
weather_conditions -> {'float64'}
road_surface_conditions -> {'float64'}
urban_or_rural_area -> {'float64'}
did_police_officer_attend_scene_of_accident -> {'float64'}
trunk_road_flag -> {'float64'}
special_conditions_at_site -> {'float64'}
carriageway_hazards_historic -> {'float64'}
carriageway_hazards -> {'float64'}
day_of_week -> {'float64'}
local_authority_district -> {'float64'}
year -> {'float64'}
month -> {'float64'}
hour -> {'float64'}


In [29]:
for col in string_columns:
    print(
        col,
        "Input types:", sample_input[col].map(type).value_counts().to_dict(),
        "Encoded types:", sample_processed[col].map(type).value_counts().to_dict()
    )

local_authority_ons_district Input types: {<class 'int'>: 1} Encoded types: {<class 'int'>: 1}
local_authority_highway_current Input types: {<class 'int'>: 1} Encoded types: {<class 'int'>: 1}
special_conditions_at_site Input types: {<class 'int'>: 1} Encoded types: {<class 'int'>: 1}
carriageway_hazards_historic Input types: {<class 'int'>: 1} Encoded types: {<class 'int'>: 1}
carriageway_hazards Input types: {<class 'int'>: 1} Encoded types: {<class 'int'>: 1}
lsoa_of_accident_location Input types: {<class 'int'>: 1} Encoded types: {<class 'int'>: 1}


In [30]:
for col in sample_processed.columns:
    print(
        col,
        "| dtype:", sample_processed[col].dtype,
        "| value:", sample_processed[col].iloc[0],
        "| Python type:", type(sample_processed[col].iloc[0])
    )

longitude | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
latitude | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
police_force | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
number_of_vehicles | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
number_of_casualties | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
day_of_week | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
local_authority_district | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
local_authority_ons_district | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
local_authority_highway_current | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
first_road_class | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
road_type | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
speed_limit | dtype: int64 | value: 0 | Python type: <class 'numpy.int64'>
junction_detail_historic | dtype: i

In [32]:
import pandas as pd

data = pd.read_csv(
    "dft-road-casualty-statistics-collision-last-5-years.csv",
    nrows=5
)

print(data.shape)
print(data.head())

(5, 44)
  collision_index  collision_year collision_ref_no  location_easting_osgr  \
0   202417H105824            2024        17H105824                 448628   
1   202517M115025            2025        17M115025                 452379   
2   202517L306225            2025        17L306225                 459273   
3   202417S104724            2024        17S104724                 440460   
4   202517S106525            2025        17S106525                 444331   

   location_northing_osgr  longitude  latitude  police_force  \
0                  533968   -1.24700  54.69840            17   
1                  519038   -1.19147  54.56387            17   
2                  523811   -1.08390  54.60600            17   
3                  517548   -1.37599  54.55157            17   
4                  518776   -1.31596  54.56228            17   

   collision_severity  number_of_vehicles  ...  carriageway_hazards_historic  \
0                   2                   2  ...                  

In [34]:
print(data[["date", "time"]].head())

         date   time
0  03/10/2024  16:06
1  29/11/2025  16:34
2  01/06/2025  22:07
3  20/04/2024  14:40
4  16/05/2025  07:25


In [35]:
data["date"] = pd.to_datetime(
    data["date"],
    format="%d/%m/%Y"
)

data["year"] = data["date"].dt.year
data["month"] = data["date"].dt.month

data["hour"] = pd.to_datetime(
    data["time"],
    format="%H:%M"
).dt.hour

print(data[["date", "time", "year", "month", "hour"]].head())

        date   time  year  month  hour
0 2024-10-03  16:06  2024     10    16
1 2025-11-29  16:34  2025     11    16
2 2025-06-01  22:07  2025      6    22
3 2024-04-20  14:40  2024      4    14
4 2025-05-16  07:25  2025      5     7


In [36]:
missing_columns = [
    col for col in feature_columns
    if col not in data.columns
]

print("Missing model features:", missing_columns)
print("Number missing:", len(missing_columns))

Missing model features: []
Number missing: 0


In [37]:
X_deploy = data[feature_columns].copy()

print("Deployment input shape:", X_deploy.shape)
print("Column order matches:", list(X_deploy.columns) == feature_columns)
print(X_deploy.head())

Deployment input shape: (5, 32)
Column order matches: True
   longitude  latitude  police_force  number_of_vehicles  \
0   -1.24700  54.69840            17                   2   
1   -1.19147  54.56387            17                   2   
2   -1.08390  54.60600            17                   2   
3   -1.37599  54.55157            17                   2   
4   -1.31596  54.56228            17                   2   

   number_of_casualties  day_of_week  local_authority_district  \
0                     1            5                        -1   
1                     1            7                        -1   
2                     3            1                        -1   
3                     2            7                        -1   
4                     1            6                        -1   

  local_authority_ons_district local_authority_highway_current  \
0                    E06000001                       E06000001   
1                    E06000002                     

In [38]:
import joblib

bundle = joblib.load("roadguard_bundle.pkl")

print("Model expected features:")
print(bundle["model"].n_features_in_)

Model expected features:
32


In [39]:
import joblib

preprocessor = joblib.load("roadguard_preprocessor.pkl")
model = joblib.load("roadguard_rf_model.pkl")

print("Preprocessor:", type(preprocessor))
print("Model:", type(model))
print("Model expected features:", model.n_features_in_)

Preprocessor: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
Model: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Model expected features: 79


In [40]:
print("Original input features:")
print(preprocessor.feature_names_in_)

print("\nNumber of original features:")
print(len(preprocessor.feature_names_in_))

Original input features:
['longitude' 'latitude' 'day_of_week' 'first_road_class' 'road_type'
 'speed_limit' 'junction_detail' 'junction_control' 'second_road_class'
 'pedestrian_crossing' 'light_conditions' 'weather_conditions'
 'road_surface_conditions' 'urban_or_rural_area' 'trunk_road_flag' 'year'
 'month' 'hour']

Number of original features:
18


In [41]:
import pandas as pd

sample_input = pd.DataFrame([{
    "longitude": -1.24700,
    "latitude": 54.69840,
    "day_of_week": 5,
    "first_road_class": 6,
    "road_type": 3,
    "speed_limit": 30,
    "junction_detail": 0,
    "junction_control": 0,
    "second_road_class": 6,
    "pedestrian_crossing": 0,
    "light_conditions": 1,
    "weather_conditions": 1,
    "road_surface_conditions": 1,
    "urban_or_rural_area": 1,
    "trunk_road_flag": 2,
    "year": 2025,
    "month": 6,
    "hour": 16
}])

processed_input = preprocessor.transform(sample_input)

prediction = model.predict(processed_input)

print("Predicted severity:", prediction[0])

Predicted severity: 3


In [42]:
target_labels = {
    1: "Fatal",
    2: "Serious",
    3: "Slight"
}

print("Predicted severity:", target_labels[prediction[0]])

Predicted severity: Slight


In [43]:
def predict_severity(input_data):

    input_df = pd.DataFrame([input_data])

    expected_features = list(preprocessor.feature_names_in_)

    missing = set(expected_features) - set(input_df.columns)

    if missing:
        raise ValueError(f"Missing features: {missing}")

    input_df = input_df[expected_features]

    processed_input = preprocessor.transform(input_df)

    prediction = model.predict(processed_input)[0]

    return target_labels[int(prediction)]

In [44]:
result = predict_severity(sample_input.iloc[0].to_dict())

print("RoadGuard AI Prediction:", result)

RoadGuard AI Prediction: Slight
